# Tutorial videos: six screen captures of the real application


Built by `scripts/tutorial_video.py`, which drives the real `Window` offscreen, grabs each frame with `QWidget.grab()`, paints the pointer, the highlight and the caption over it, and encodes the frames with `/usr/bin/ffmpeg`.



Nothing in the clips is staged. Every panel is the real one, every number was computed by the shipped code while the clip was recorded, and the only pixels the script paints are the overlay. This notebook measures what came out.



    QT_QPA_PLATFORM=offscreen python scripts/tutorial_video.py

In [1]:
import json, os, subprocess
import pandas as pd
ROOT = '/media/carruthers/mnt3/claude/repo/starplast/.claude/worktrees/agent-a0225e1f3ae74e85e'
OUT = os.path.join(ROOT, 'docs', 'tutorial', 'video')
sorted(os.listdir(OUT))

['1_find_a_gene_poster.jpg', '1_find_a_gene_silent.mp4', '2_start_here_poster.jpg', '2_start_here_silent.mp4', '3_test_before_you_trust_poster.jpg', '3_test_before_you_trust_silent.mp4', '4_pick_a_map_poster.jpg', '4_pick_a_map_silent.mp4', '5_walk_the_network_poster.jpg', '5_walk_the_network_silent.mp4', '6_one_question_poster.jpg', '6_one_question_silent.mp4']

## What each clip is, and what it came to



`ffprobe` on the committed files: the duration, the frame rate and the codec are what a browser will actually play, not what the encoder was asked for.

In [2]:
def probe(path):
    out = subprocess.run(['/usr/bin/ffprobe', '-v', 'error', '-show_entries',
                          'stream=width,height,r_frame_rate,codec_name,pix_fmt',
                          '-show_entries', 'format=duration', '-of', 'json', path],
                         capture_output=True, text=True, check=True)
    d = json.loads(out.stdout)
    s = d['streams'][0]
    return {'codec': s['codec_name'], 'size': f"{s['width']}x{s['height']}",
            'fps': s['r_frame_rate'], 'pix_fmt': s['pix_fmt'],
            'seconds': round(float(d['format']['duration']), 1)}

DESCRIPTIONS = {"1_find_a_gene": ["Find a gene and read its evidence", "The find box, the evidence panel, and the rule that a dash is not a zero.", "1_explore"], "2_start_here": ["Start here, end to end", "The guided tab: what you have, which genes, which label, what you want to know -- then the ranked strategies and Run.", "3_gene_list"], "3_test_before_you_trust": ["Test before you trust", "A strategy card, its four bars, the hold-out test, and the verdict with its chance level.", "4_test_and_calibrate"], "4_pick_a_map": ["Pick a map that shows your label", "The pregenerated gallery, sorted by how well a label maps onto each, and the score table that says so.", "2_holdout_search"], "5_walk_the_network": ["Walk the network from one gene", "The star map: one gene's measured links by source, widened to its neighbourhood and re-centred on a neighbour.", "5_networks_and_agreement"], "6_one_question": ["One real biological question, answered", "Question 13 of instruction 59: which never-published proteins get a confident compartment call, from the question to the named genes.", "6_advanced_models"]}
rows = []
for slug, (title, what, tutorial) in sorted(DESCRIPTIONS.items()):
    video = os.path.join(OUT, f'{slug}_silent.mp4')
    poster = os.path.join(OUT, f'{slug}_poster.jpg')
    rows.append({'clip': slug, 'title': title, **probe(video),
                 'video_kB': round(os.path.getsize(video) / 1000),
                 'poster_kB': round(os.path.getsize(poster) / 1000),
                 'tutorial': tutorial})
clips = pd.DataFrame(rows)
clips

,clip,title,codec,size,fps,pix_fmt,seconds,video_kB,poster_kB,tutorial
0,1_find_a_gene,Find a gene and read its evidence,h264,1680x1050,12/1,yuv420p,60.3,759,83,1_explore
1,2_start_here,"Start here, end to end",h264,1680x1050,12/1,yuv420p,63.4,1301,104,3_gene_list
2,3_test_before_you_trust,Test before you trust,h264,1680x1050,12/1,yuv420p,61.6,821,100,4_test_and_calibrate
3,4_pick_a_map,Pick a map that shows your label,h264,1680x1050,12/1,yuv420p,62.2,689,128,2_holdout_search
4,5_walk_the_network,Walk the network from one gene,h264,1680x1050,12/1,yuv420p,64.6,1026,114,5_networks_and_agreement
5,6_one_question,"One real biological question, answered",h264,1680x1050,12/1,yuv420p,62,826,116,6_advanced_models


In [3]:
added = sum(os.path.getsize(os.path.join(OUT, f)) for f in os.listdir(OUT))
print(f'{len(os.listdir(OUT))} files, {added:,} bytes ({added / 1e6:.1f} MB) added to the repository')
print('every clip between 60 and 120 s:', bool((clips.seconds.between(60, 120)).all()))

12 files, 6,068,077 bytes (6.1 MB) added to the repository
every clip between 60 and 120 s: True


## Clip 6 shows the real answer to a real question



The sixth clip runs question 13 of `instructions/open/59_biological_questions.md` -- which never-published proteins get a confident compartment call -- live in the Strategies tab. The genes its caption names must be the genes the recorded run of that question produced, or the clip is showing something made up.

In [4]:
recorded = pd.read_csv(os.path.join(ROOT, 'results', 'questions_2026_09_30',
                                    'Q13_understudied_first_candidates.csv'))
recorded.head(3)[['gene_id', 'product', 'prediction', 'support', 'sources_agreeing']]

,gene_id,product,prediction,support,sources_agreeing
0,TGME49_209000,HECT-domain (ubiquitin-transferase) domain-containing protein,nucleus - chromatin,0.546,3
1,TGME49_225745,hypothetical protein,nucleus - chromatin,0.546,3
2,TGME49_266010,phosphatidylinositol 3- and 4-kinase,nucleus - chromatin,0.546,3


In [5]:
named_in_the_clip = ['TGME49_209000', 'TGME49_225745', 'TGME49_266010']
print('the clip names:', named_in_the_clip)
print('the recorded run:', list(recorded.gene_id.head(3)))
print('match:', list(recorded.gene_id.head(3)) == named_in_the_clip)
print('candidates in the recorded run:', len(recorded))

the clip names: ['TGME49_209000', 'TGME49_225745', 'TGME49_266010']
the recorded run: ['TGME49_209000', 'TGME49_225745', 'TGME49_266010']
match: True
candidates in the recorded run: 1723


## The clips ship with the site, not with the wheel



`scripts/build_docs.py` copies `docs/tutorial/` -- the written tutorials and these clips -- into `docs/site/`. The Python package declares its own `packages` and `package-data`, neither of which reaches `docs/`, so nothing here is installed by pip.

In [6]:
import tomllib
with open(os.path.join(ROOT, 'pyproject.toml'), 'rb') as fh:
    cfg = tomllib.load(fh)['tool']['setuptools']
print('packages:', cfg['packages'])
print('package data:', cfg['package-data'])
print('include-package-data:', cfg['include-package-data'])

packages: ['starplast', 'starplast.spaces']
package data: {'starplast': ['data/*.npz', 'data/*.parquet', 'data/*.tsv', 'data/*.tsv.gz', 'data/*.json', 'data/hf_release/*.parquet', 'data/hf_release/*.md', 'data/icons/*.svg']}
include-package-data: False


## Rebuilding



    QT_QPA_PLATFORM=offscreen python scripts/tutorial_video.py             # all six

    QT_QPA_PLATFORM=offscreen python scripts/tutorial_video.py 4_pick_a_map  # one

    python scripts/tutorial_video.py --pages                               # relink only



`scripts/build_tutorials.py` calls `publish_pages()` at the end, so regenerating the written tutorials puts the clips back into them rather than dropping them.